In [11]:
import hydra
import yaml
from omegaconf import DictConfig
import pytorch_lightning as pl
import os
import sys
import pickle

from pytorch_lightning.callbacks import LearningRateMonitor
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers import TensorBoardLogger

with open('ptls_config.yaml', 'rb') as fl:
    conf = DictConfig(yaml.safe_load(fl))

In [3]:
with open('../data/train_trx.parquet', 'rb') as fl:
    data = pickle.load(fl)

In [6]:
dm = hydra.utils.instantiate(conf.data_module)

In [12]:
#train
if 'seed_everything' in conf:
    pl.seed_everything(conf.seed_everything)

model = hydra.utils.instantiate(conf.pl_module)
#dm = hydra.utils.instantiate(conf.data_module)

_trainer_params = conf.trainer
_trainer_params_additional = {}

_trainer_params_callbacks = []

lr_monitor = LearningRateMonitor(logging_interval='step')
_trainer_params_callbacks.append(lr_monitor)

if len(_trainer_params_callbacks) > 0:
    _trainer_params_additional['callbacks'] = _trainer_params_callbacks
    
trainer = pl.Trainer(**_trainer_params, **_trainer_params_additional)
trainer.fit(model, dm)

if 'model_path' in conf:
    torch.save(model.seq_encoder, conf.model_path)

Global seed set to 42
/root/anaconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:478: LightningDeprecationWarning: Setting `Trainer(gpus=0)` is deprecated in v1.7 and will be removed in v2.0. Please use `Trainer(accelerator='gpu', devices=0)` instead.
  rank_zero_deprecation(
/root/anaconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/accelerator_connector.py:589: LightningDeprecationWarning: The Trainer argument `auto_select_gpus` has been deprecated in v1.9.0 and will be removed in v2.0.0. Please use the function `pytorch_lightning.accelerators.find_usable_cuda_devices` instead.
  rank_zero_deprecation(
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
Missing logger folder: /home/everdark/diploma/ptls-glove/lightning_logs

  | Name               | Type            | Params
-------------------------

Sanity Checking: 0it [00:00, ?it/s]

/root/anaconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, val_dataloader 0, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(
/root/anaconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:224: PossibleUserWarning: The dataloader, train_dataloader, does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` (try 16 which is the number of cpus on this machine) in the `DataLoader` init to improve performance.
  rank_zero_warn(


Training: 0it [00:00, ?it/s]

/root/anaconda3/envs/torch/lib/python3.10/site-packages/pytorch_lightning/trainer/call.py:48: UserWarning: Detected KeyboardInterrupt, attempting graceful shutdown...
  rank_zero_warn("Detected KeyboardInterrupt, attempting graceful shutdown...")


NameError: name 'torch' is not defined

In [ ]:
#inference

def save_scores(df_scores, output_conf):
    output_name = output_conf.path
    output_format = output_conf.format

    output_path = f'{output_name}.{output_format}'

    if output_format == 'pickle':
        df_scores.to_pickle(output_path)
    elif output_format == 'csv':
        df_scores.to_csv(output_path, sep=',', header=True, index=False)
    elif output_format == 'parquet':
        df_scores.to_parquet(output_path)
    else:
        print('error')


if 'seed_everything' in conf:
    pl.seed_everything(conf.seed_everything)

seq_encoder = hydra.utils.instantiate(conf.inference.seq_encoder)
if type(seq_encoder) is DictConfig and seq_encoder.get('load_from_checkpoint', False):
    pl_module = hydra.utils.instantiate(conf.pl_module)
    pl_module.load_state_dict(torch.load(seq_encoder['f'])['state_dict'])
    seq_encoder = pl_module.seq_encoder
model = InferenceModule(
    model=seq_encoder,
    pandas_output=True, model_out_name='emb',
)
model.model.is_reduce_sequence = True

dataset_inference = hydra.utils.instantiate(conf.inference.dataset)
inference_dl = DataLoader(
    dataset=dataset_inference,
    collate_fn=collate_feature_dict,
    shuffle=False,
    num_workers=conf.inference.get('num_workers', 0),
    batch_size=conf.inference.get('batch_size', 128),
)

if torch.cuda.is_available():
    accelerator = "cuda"
    devices = 1
else:
    accelerator = "cpu"
    devices = 0

df_scores = pl.Trainer(accelerator=accelerator, devices=devices, max_epochs=-1).predict(model, inference_dl)
df_scores = pd.concat(df_scores, axis=0)

save_scores(df_scores, conf.inference.output)

In [ ]:
#boosting validation:

